In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# ============================================================
# Pairwise Net Spillover
# ------------------------------------------------------------
# 입력
#   - gfevd_all.npy : shape (T_eff, N, N)
#     row = response variable
#     col = shock source
#
# 정의
#   pairwise_net_ij(t) = theta[j, i] - theta[i, j]
#   > 0  : i -> j 방향 우세
#   < 0  : j -> i 방향 우세
#
# 출력
#   - ./pairwise_output/pairwise_net_all.npy
#   - ./pairwise_output/pairwise_net_mean.csv
#   - ./pairwise_output/pairwise_net_long.csv
#   - ./pairwise_output/pairwise_net_heatmap.png
#   - ./pairwise_output/pairwise_net_heatmap_interactive.html
#
# 선택 출력
#   - 특정 pair 시계열 png/html
# ============================================================

# =========================
# Config
# =========================
BASE_DIR = Path("./")
OUT_DIR = BASE_DIR / "pairwise_output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GFEVD_FILE = BASE_DIR / "gfevd_all.npy"

DATE_FILE_CANDIDATES = [
    BASE_DIR / "tvpvar_input_scaled.csv",
    BASE_DIR / "tvpvar_input_transformed.csv",
    BASE_DIR / "tvpvar_raw_level_merged.csv",
]

VAR_NAMES = [
    "SOLVPN",
    "COPPER",
    "GOLD",
    "SILVER",
    "DXY",
    "UST10Y",
    "VIX"
]

# 특정 pair 시계열도 같이 저장할지
SAVE_SELECTED_PAIR_SERIES = True
SELECTED_PAIRS = [
    ("SOLVPN", "COPPER"),
    ("COPPER", "SOLVPN"),
    ("SOLVPN", "GOLD"),
    ("SOLVPN", "VIX"),
]

OUT_ALL = OUT_DIR / "pairwise_net_all.npy"
OUT_MEAN = OUT_DIR / "pairwise_net_mean.csv"
OUT_LONG = OUT_DIR / "pairwise_net_long.csv"
OUT_HEATMAP = OUT_DIR / "pairwise_net_heatmap.png"
OUT_HEATMAP_HTML = OUT_DIR / "pairwise_net_heatmap_interactive.html"

ROW_SUM_TOL = 1e-6
FORCE_ROW_NORMALIZE = True

# =========================
# Helper
# =========================
def load_dates(target_len: int):
    for fp in DATE_FILE_CANDIDATES:
        if fp.exists():
            try:
                df = pd.read_csv(fp)
                if "Date" not in df.columns:
                    continue

                dt = pd.to_datetime(df["Date"], errors="coerce").dropna().reset_index(drop=True)

                if len(dt) < target_len:
                    continue

                dt = dt.iloc[-target_len:].reset_index(drop=True)
                return dt

            except Exception as e:
                print(f"[WARN] Failed to read dates from {fp}: {e}")

    return None


def row_normalize_theta(theta: np.ndarray) -> np.ndarray:
    row_sum = theta.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1.0
    return theta / row_sum


def compute_pairwise_net(theta: np.ndarray) -> np.ndarray:
    """
    theta: (N, N)
    row = response variable
    col = shock source

    pairwise_net[i, j] > 0  => i -> j 우세
    pairwise_net[i, j] < 0  => j -> i 우세
    """
    n = theta.shape[0]
    out = np.zeros((n, n), dtype=float)

    for i in range(n):
        for j in range(n):
            if i == j:
                out[i, j] = 0.0
            else:
                out[i, j] = theta[j, i] - theta[i, j]

    return out


def save_pair_series(df_time: pd.DataFrame, x_col: str, pair_name: str, out_png: Path, out_html: Path):
    plt.figure(figsize=(12, 5))
    plt.plot(df_time[x_col], df_time[pair_name], linewidth=1.3)
    plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.title(f"Pairwise Net Spillover: {pair_name}")
    plt.xlabel(x_col)
    plt.ylabel("Pairwise NET (%)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=df_time[x_col],
            y=df_time[pair_name],
            mode="lines",
            name=pair_name
        )
    )
    fig.add_hline(y=0.0)
    fig.update_layout(
        title=f"Interactive Pairwise Net Spillover: {pair_name}",
        xaxis_title=x_col,
        yaxis_title="Pairwise NET (%)",
        height=500,
        hovermode="x unified"
    )
    fig.show()
    fig.write_html(str(out_html))


# =========================
# Load
# =========================
if not GFEVD_FILE.exists():
    raise FileNotFoundError(f"Not found: {GFEVD_FILE}")

gfevd_all = np.load(GFEVD_FILE)

if gfevd_all.ndim != 3:
    raise ValueError(f"gfevd_all must be 3D, got shape={gfevd_all.shape}")

T_eff, N, N2 = gfevd_all.shape

if N != N2:
    raise ValueError(f"Last two dims must be square, got shape={gfevd_all.shape}")

if len(VAR_NAMES) != N:
    raise ValueError(f"len(VAR_NAMES)={len(VAR_NAMES)} must match N={N}")

print(f"[INFO] Loaded GFEVD: shape={gfevd_all.shape}")

# =========================
# Check / Normalize
# =========================
gfevd_proc = gfevd_all.copy()

row_sum_check = gfevd_proc.sum(axis=2)
max_row_err = np.max(np.abs(row_sum_check - 1.0))
print(f"[INFO] Max row-sum error before normalization: {max_row_err:.8f}")

if FORCE_ROW_NORMALIZE and max_row_err > ROW_SUM_TOL:
    print("[INFO] Applying row normalization to all theta_t")
    for t in range(T_eff):
        gfevd_proc[t] = row_normalize_theta(gfevd_proc[t])

row_sum_check_after = gfevd_proc.sum(axis=2)
max_row_err_after = np.max(np.abs(row_sum_check_after - 1.0))
print(f"[INFO] Max row-sum error after normalization: {max_row_err_after:.8f}")

# =========================
# Dates
# =========================
dates = load_dates(T_eff)

# =========================
# Pairwise Net for all t
# =========================
pairwise_all = np.zeros((T_eff, N, N), dtype=float)

for t in range(T_eff):
    pairwise_all[t] = compute_pairwise_net(gfevd_proc[t])

pairwise_mean = pairwise_all.mean(axis=0) * 100.0
df_mean = pd.DataFrame(pairwise_mean, index=VAR_NAMES, columns=VAR_NAMES)

# =========================
# Long format time series
# =========================
records = []

for t in range(T_eff):
    for i, src in enumerate(VAR_NAMES):
        for j, dst in enumerate(VAR_NAMES):
            if i == j:
                continue

            row = {
                "Source": src,
                "Target": dst,
                "Pairwise_NET": pairwise_all[t, i, j] * 100.0
            }

            if dates is not None:
                row["Date"] = dates.iloc[t]
            else:
                row["t"] = t

            records.append(row)

df_long = pd.DataFrame(records)

# wide format for selected pairs
if dates is not None:
    df_time = pd.DataFrame({"Date": dates})
    x_col = "Date"
else:
    df_time = pd.DataFrame({"t": np.arange(T_eff)})
    x_col = "t"

for i, src in enumerate(VAR_NAMES):
    for j, dst in enumerate(VAR_NAMES):
        if i == j:
            continue
        pair_name = f"{src}_to_{dst}"
        df_time[pair_name] = pairwise_all[:, i, j] * 100.0

# =========================
# Save
# =========================
np.save(OUT_ALL, pairwise_all)
df_mean.to_csv(OUT_MEAN, encoding="utf-8-sig")
df_long.to_csv(OUT_LONG, index=False, encoding="utf-8-sig")

print("[INFO] Saved:")
print(f"  - {OUT_ALL}")
print(f"  - {OUT_MEAN}")
print(f"  - {OUT_LONG}")

# =========================
# Static Heatmap (matplotlib)
# =========================
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(df_mean.values)

ax.set_xticks(np.arange(N))
ax.set_yticks(np.arange(N))
ax.set_xticklabels(VAR_NAMES, rotation=45, ha="right")
ax.set_yticklabels(VAR_NAMES)

for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{df_mean.values[i, j]:.2f}", ha="center", va="center")

ax.set_title("Mean Pairwise Net Spillover")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig(OUT_HEATMAP, dpi=300, bbox_inches="tight")
plt.close()

# =========================
# Interactive Heatmap (plotly)
# =========================
fig = go.Figure(
    data=go.Heatmap(
        z=df_mean.values,
        x=VAR_NAMES,
        y=VAR_NAMES,
        text=np.round(df_mean.values, 2),
        texttemplate="%{text}",
        hovertemplate="Source=%{y}<br>Target=%{x}<br>Pairwise NET=%{z:.2f}%<extra></extra>"
    )
)

fig.update_layout(
    title="Interactive Mean Pairwise Net Spillover",
    xaxis_title="Target",
    yaxis_title="Source",
    height=650
)

fig.show()
fig.write_html(str(OUT_HEATMAP_HTML))

print(f"  - {OUT_HEATMAP}")
print(f"  - {OUT_HEATMAP_HTML}")

# =========================
# Selected pair series
# =========================
if SAVE_SELECTED_PAIR_SERIES:
    for src, dst in SELECTED_PAIRS:
        if src not in VAR_NAMES or dst not in VAR_NAMES or src == dst:
            print(f"[WARN] Skip invalid pair: {src}, {dst}")
            continue

        pair_name = f"{src}_to_{dst}"
        out_png = OUT_DIR / f"{pair_name}.png"
        out_html = OUT_DIR / f"{pair_name}.html"

        save_pair_series(df_time, x_col, pair_name, out_png, out_html)

        print(f"  - {out_png}")
        print(f"  - {out_html}")

[INFO] Loaded GFEVD: shape=(1035, 7, 7)
[INFO] Max row-sum error before normalization: 0.00000000
[INFO] Max row-sum error after normalization: 0.00000000
[INFO] Saved:
  - pairwise_output\pairwise_net_all.npy
  - pairwise_output\pairwise_net_mean.csv
  - pairwise_output\pairwise_net_long.csv


  - pairwise_output\pairwise_net_heatmap.png
  - pairwise_output\pairwise_net_heatmap_interactive.html


  - pairwise_output\SOLVPN_to_COPPER.png
  - pairwise_output\SOLVPN_to_COPPER.html


  - pairwise_output\COPPER_to_SOLVPN.png
  - pairwise_output\COPPER_to_SOLVPN.html


  - pairwise_output\SOLVPN_to_GOLD.png
  - pairwise_output\SOLVPN_to_GOLD.html


  - pairwise_output\SOLVPN_to_VIX.png
  - pairwise_output\SOLVPN_to_VIX.html
